In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# 파일 경로 설정부
file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_view_delete\Membership_v2.csv"

# 사용 컬럼 설정부
use_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "is_repurchase",
]

# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)

# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

# 데이터 로드부
df = pd.read_csv(file_path, usecols=use_cols).copy()

# 숫자형 변환부
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[
    [
        "price",
        "max_screen",
        "is_promotion",
        "is_churn_prevented",
        "payment_device",
        "is_user_verified",
        "gender",
        "age",
    ]
].copy()

# 양성 클래스 정의부
# is_repurchase == 0 을 예측 목표로 두기 때문에 0이면 1, 1이면 0으로 변환
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
numeric_features = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
    "age",
]

categorical_features = [
    "payment_device",
    "gender",
]

# 전처리 파이프라인 구성부
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 모델 정의부
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42,
    ),
}

# 평가 수행부
results = []

for model_name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    result = {
        "model": model_name,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

    results.append(result)

# 결과 출력부
results_df = (
    pd.DataFrame(results)
    .set_index("model")
    [["precision", "recall", "f1_score", "roc_auc", "pr_auc"]]
    .round(4)
    .sort_values("f1_score", ascending=False)
)

print("양성 클래스 기준: is_repurchase == 0")
print(results_df)


양성 클래스 기준: is_repurchase == 0
                    precision  recall  f1_score  roc_auc  pr_auc
model                                                           
LogisticRegression     0.3379  0.5768    0.4261   0.5800  0.3393
RandomForest           0.3337  0.5196    0.4064   0.5615  0.3290
GradientBoosting       0.3333  0.0008    0.0015   0.5873  0.3460
